# 03 — Sampler dynamics：Full-space、Fixed-cardinality、Effective Exchange 与 Reverse Annealing

## 当前定位

这一阶段的研究目标是：

$$
\boxed{\text{尽量保持问题 Hamiltonian 不变，只优化 sampler dynamics}}
$$

当前优先 full-space；hard fixed-cardinality 作为重要对照；reverse annealing 暂缓到后续。

## 1. Full-space 与 fixed-cardinality 双路线

### Full-space（优先）

$$
\Omega=\{0,1\}^{261}.
$$

保留：

$$
H_{\mathrm{count}}
=
A_{\mathrm{count}}
\left(\sum_e x_e-143\right)^2.
$$

允许 cardinality 改变。

只要保留 single-bit channel，整个 binary space 仍可达。

### Fixed-cardinality（对照）

$$
\Omega_{143}
=
\left\{x:\sum_e x_e=143\right\}.
$$

此时：

$$
H_{\mathrm{count}}\equiv0.
$$

可删掉 dense count square。

优势是搜索几何更贴近 Hamilton path；缺点是这不是 standard independent transverse-field driver 的完整状态空间。

## 2. Kernel 命名

### Exchange move size：$k_{\mathrm{ex}}$

$$
k_{\mathrm{ex}}\text{ 个 selected edges 关闭}
$$

同时：

$$
k_{\mathrm{ex}}\text{ 个 unselected edges 打开}.
$$

因此：

$$
\Delta|E_x|=0.
$$

### Reconnect

当前为 directed 2-switch：

$$
(a\to b,c\to d)
\leftrightarrow
(a\to d,c\to b).
$$

这是独立于 $k_{\mathrm{ex}}$ 的 4-bit topology move。

### Insertion / removal

当前测试了：

$$
\{a\to b\}
\leftrightarrow
\{a\to c,c\to b\}.
$$

它是一个 3-bit move：

$$
\Delta |E_x|=\pm1.
$$

它属于 full-space cardinality-changing kernel。

## 3. 为什么 reconnect 值得比随机 $k_{\mathrm{ex}}>1$ 更优先优化

随机 $k_{\mathrm{ex}}=2,3$ 的接受率明显低于 $k=1$，即使后期增加多次独立尝试，也没有稳定改善。

而 reconnect 有不同性质：

$$
d_v^{in},d_v^{out}
$$

完全保持。

因此在 degree 已经正确以后，它可以直接改变 cycle / component topology，而不重新破坏 degree。

我们已经有一个真实成功轨迹：

$$
1{,}326{,}008
\xrightarrow{\text{late reconnect}}
1{,}343{,}093.
$$

所以接下来更值得做：

- reconnect intensity；
- 3-edge reconnect；
- cycle-boundary reconnect；
- component-endpoint reconnect；

而不是继续单纯增加随机 $k_{\mathrm{ex}}=3$ 次数。

## 4. Dynamic kernel mixture

若每个子 kernel 本身满足：

$$
q_i(x\to x')=q_i(x'\to x),
$$

且 mixture 权重只依赖时间：

$$
p_i=p_i(t),
$$

则：

$$
q_t(x\to x')
=
\sum_i p_i(t)q_i(x\to x')
$$

仍然对称。

因此可以做 anneal-time / outer-iteration dependent schedule。

但如果：

$$
p_i=p_i(x),
$$

例如“degree conflict 小于 2 就增加 reconnect”，proposal 一般不再自动对称，需要显式 Metropolis-Hastings correction。

## 5. Multi-bit proposal 是否仍算标准 SQA？

关键不是“是否一次翻多个 bits”。

只要 full-space 不变，并且 sampler 仍针对同一个 Suzuki–Trotter effective distribution 做 detailed-balance MCMC，multi-bit / cluster proposal 可以只是采样加速。

因此：

$$
\boxed{\text{multi-bit MCMC}\neq\text{自动改变 quantum driver}}
$$

真正与标准 independent transverse field 不一致的是：

$$
\boxed{\text{把状态空间硬限制在 fixed Hamming weight sector}}
$$

因为：

$$
-\Gamma\sum_i\sigma_i^x
$$

会单独翻一个 bit，从而改变 Hamming weight。

## 6. Effective second-order exchange：为什么两条路线并非毫无关系

考虑 full-space 中强 count penalty：

$$
H_P
\supset
A_{\mathrm{count}}
\left(\sum_i x_i-K\right)^2
$$

以及标准 transverse field：

$$
H_D=-\Gamma\sum_i\sigma_i^x.
$$

在 $K$-edge 低能 sector 中，一个单 bit flip 会暂时离开该 sector：

$$
10\to00
$$

或：

$$
10\to11.
$$

第二次 flip 可回到：

$$
01.
$$

所以低能有效动力学中会出现 exchange-like：

$$
10\leftrightarrow01.
$$

量级预期类似：

$$
J_{\mathrm{eff}}
\sim
\frac{\Gamma^2}{\Delta_{\mathrm{count}}},
$$

但这目前只是物理图像，还没有做正式 Schrieffer–Wolff / degenerate perturbation derivation。

意义是：

- full-space：exchange 通过高能 virtual states 间接发生；
- fixed-cardinality sampler：直接把 exchange 当作一次 proposal。

这为两条路线提供了形式联系。

## 7. Full-space 第一轮实验带来的新问题

full-space + count penalty 后，sampler 很容易达到：

$$
R_{\deg}=0,
$$

但 selected edges 常停在：

$$
139\sim142
$$

而不是 143。

原因是少选边有利于消除 degree penalty；一旦成为 under-filled path cover，balanced exchange / reconnect 都满足：

$$
\Delta|E_x|=0
$$

无法补回缺边。

single-bit addition 又可能需要先制造高能 degree/count 中间态。

因此 full-space 的核心问题是：

$$
\boxed{\text{cardinality + degree 联合 barrier}}
$$

## 8. Insertion/removal 的第一版结果

测试 move：

$$
\{a\to b\}
\leftrightarrow
\{a\to c,c\to b\}.
$$

complex144 中有 134 个 static templates。

优点：

- proposal 可设计成 involution；
- 能改变 cardinality；
- 比随机 single addition 更结构化。

观察：

- eligibility 约 64%；
- conditional acceptance 仍非常低；
- $A_{\mathrm{count}}=32$ 与 64 都未稳定得到 143-edge Hamilton path。

因此下一步应从简单 1↔2 扩展成更能同时修结构的：

$$
k\leftrightarrow k+1
$$

局部 reroute。

## 9. Reverse annealing：记录想法，暂缓实验

当前 projected SQA 即使 warm-start 上一轮结果，仍是：

$$
\Gamma:\text{high}\to\text{low},
$$

因此是 warm-started forward SQA，不是 reverse annealing。

真正 reverse annealing 从已知 classical state 出发：

$$
\Gamma\approx0
\to
\Gamma_{\mathrm{peak}}
\to
0.
$$

它最适合：

1. 已有合法 Hamilton path，做 weighted refinement；
2. forward search 反复回到同一 basin；
3. 多个 feasible seed 的局部 refinement；
4. 研究 complex144 多-path basin 之间是否可跨越。

该方向暂缓，待 full-space forward sampler 更清楚以后再单独做。

## 10. 当前实验路线

### 优先

$$
\boxed{\text{Full-space + structured multi-bit kernels}}
$$

重点：

- single-bit 保证全空间可达；
- exchange 做近 fixed-cardinality motion；
- reconnect 做 topology change；
- insertion/removal 或 $k\leftrightarrow k+1$ 做 cardinality repair。

### 对照

$$
\boxed{\text{Hard fixed-cardinality}}
$$

用于量化“直接把 hard constraint 写进 sampler geometry”能获得多少性能收益。

### 暂缓

- reverse annealing；
- compression 作为默认前处理；
- 继续盲目增加随机 $k_{\mathrm{ex}}>1$。

## 11. 最新补充：单纯增加 reconnect 次数也不够

为确认“真正值得优化的是 reconnect”这一点，我们测试了一个最简单的 reconnect 强度实验。

前 12 个 outer iterations 仍使用 $k_{\mathrm{ex}}=1$ exchange 修 degree；后 6 个 iterations 中，每个 slice/sweep 在一次 exchange 之外额外执行：

- 1 次 reconnect；
- 2 次 reconnect；
- 4 次 reconnect。

每个 reconnect 都是独立的 symmetric Metropolis transition，不做 best-of-many 选择。

4 seeds/config 的结果中，三组都没有得到 Hamilton path；最低 degree conflict 都只能到 1 左右。增加 reconnect 次数把 native proposal 数从约 46 万提高到约 81 万，但没有带来明显突破。

因此当前结论不是“reconnect 没用”。此前已有一次 reconnect phase 把

$$
1{,}326{,}008
\to
1{,}343{,}093.
$$

真正说明的是：

$$
\boxed{\text{reconnect 的价值在于结构，而不是单纯多试几次同一个 2-switch。}}
$$

下一步更值得测试的是：

- 3-edge reconnect；
- cycle-boundary reconnect；
- component-endpoint reconnect；
- 在 degree 已经为 0 时专门生成能打断 cycle / 合并 component 的 reconnect。

这和随机增加 $k_{\mathrm{ex}}>1$ 次数得到的结论一致：**需要更有结构的联合 move，而不是更多无条件随机 move。**

## 12. Reconnect 的几何结构与“良好 move 很稀少”诊断

对于 fixed-cardinality 状态：

$$
|E_x|=N-1,
$$

若同时满足

$$
d_v^{in}\le1,\qquad d_v^{out}\le1,
$$

则 selected graph 的每个弱连通分量只能是 directed path 或 directed cycle。

设 path components 数为 $p$、cycle components 数为 $c$。由于每个 path component 比顶点数少一条边，而 cycle component 的边数等于顶点数：

$$
|E_x|=N-p.
$$

当 $|E_x|=N-1$ 时：

$$
\boxed{p=1}.
$$

因此 degree-correct 的 143-edge 状态一定是：

$$
\boxed{\text{一条 open path }+\text{若干 cycles}.}
$$

这给 reconnect 一个非常清楚的几何解释。

当前 2-edge reconnect：

$$
(a\to b,c\to d)
\leftrightarrow
(a\to d,c\to b)
$$

根据两条被删除 selected edges 所在的 component，可产生：

- path + path（实际上是同一条 open path 上的两条边）：通常把一段闭合成 cycle，因此 $c\to c+1$；
- path + cycle：若 cross edges 存在，可把该 cycle 吸收到 open path，$c\to c-1$；
- 两个不同 cycles：可把两个 cycle 合成一个，$c\to c-1$；
- 同一个 cycle 内两条边：可把一个 cycle 分成两个，$c\to c+1$。

这解释了此前成功轨迹：

$$
\text{Hamilton path}
\to
\text{path + cycles}
\to
\text{另一 Hamilton path}.
$$

也说明 reconnect 的主要价值不是“直接在 Hamilton paths 之间跳”，而是允许经过 degree-correct 的 cycle intermediate states。

### 当前数据的关键诊断

对长预算实验的 checkpoint 重新完整枚举 41 个 static 2-switch templates：

- 一个 seed 在 exchange phase 后已经是 score $1{,}326{,}008$ 的 Hamilton path。8 个 reconnect template eligible，其中有 3 个在下一轮 classical BQM 上是 downhill，但 **没有任何一个 reconnect 后仍是 Hamilton path**。也就是说，从 Hamilton path 出发，单个 reconnect 的合理作用就是先离开 Hamilton manifold。
- 一个 seed 有 degree conflict 1、2 个 cycles。9 个 template eligible，其中 6 个可减少一个 cycle；这里随机抽 eligible reconnect 的“结构良好”概率其实并不低。
- 最重要的一个 stuck seed 满足：

$$
|E_x|=143,\qquad R_{deg}=0,\qquad c=1.
$$

它有 8 个 eligible 2-switch reconnect，其中 4 个在 classical BQM 上甚至是 downhill，但：

$$
\boxed{0\text{ 个可以减少 cycle}}
$$

因此“降低能量的 move”与“朝 Hamilton feasibility 前进的 move”不是同一概念。

随后对该 stuck state 穷举所有真正的 3-edge degree-preserving reconnect（保持同一 tail/head 集，仅重新排列 3 个 head），结果仍然是：

$$
\boxed{0\text{ 个 cycle-reducing order-3 reconnect}}
$$

所以这个状态不是“良好 reconnect 太难随机到”，而是当前 $r=2,3$ reconnect neighborhood 中 **根本不存在** 所需 repair move。

这直接解释了为什么把随机抽取次数提高 2--3 倍没有作用。

### Matching / alternating-cycle 视角

当 degree conflict 为 0 时，可把 selected edges 看成 tail-copy 与 head-copy 之间的 matching。

保持所有顶点 in/out degree 不变的 $r$-edge reconnect，本质上是沿 bipartite candidate graph 中一个长度 $2r$ 的 alternating cycle 翻转 matching。

因此：

- 当前 2-switch 是 4-cycle flip；
- 3-edge reconnect 是 6-cycle flip；
- 若 sparse OLC graph 中不存在相应短 alternating cycle，就必须使用更长 alternating-cycle move，或允许 endpoint / degree / cardinality 暂时改变。

另外，纯 degree-preserving reconnect 会固定 unmatched tail/head，因此固定当前 source/sink。若希望在不同 endpoint 的 Hamilton paths 之间移动，还需要 alternating-path / endpoint-shift 类 move，或 full-space 中暂时改变 degree/cardinality。

### 对“提高抽中良好 move 概率”的结论

必须先区分：

1. **良好 move 存在但稀少**：优化 proposal distribution 有意义；
2. **当前 neighborhood 中良好 move 数为 0**：再多抽样也无效，必须扩展 move class。

若良好 move 存在，可以在每个 slice 直接枚举当前所有 eligible reconnect（当前只有几十个），根据完整 path-integral action change $\Delta S$ 和结构信息构造 informed proposal：

$$
q(x\to y_m)
\propto
w_m(x).
$$

因为这种 proposal 是 state-dependent，要保持同一 SQA target，应使用：

$$
A(x\to y)
=
\min\left[
1,
e^{-\Delta S}
\frac{q(y\to x)}{q(x\to y)}
\right].
$$

这样可以显著提高抽中有意义 move 的概率，而不必退化成“随机抽很多次然后选最好”的非平衡 heuristic。

下一步应优先研究：

- 更长 alternating-cycle reconnect；
- endpoint-shift / alternating-path move；
- cycle/component-aware proposal；
- locally-balanced / Metropolis-Hastings guided reconnect。

